In [2]:
import os
import re
import unicodedata
import random
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

# =========================
# 1. Config
# =========================
CSV_PATH = "/kaggle/input/datasets/infinityvoid123/all-train/all_train_dataset_final.csv"
TEXT_COL = "bangla_speech"
LABEL_COL = "dialect"
RANDOM_STATE = 42
MAX_LEN = 128
EPOCHS = 5
LR = 2e-5
BATCH_SIZE = 16
MODEL_NAME = "csebuetnlp/banglabert"   # mBERT: google-bert/bert-base-multilingual-cased
OUTPUT_DIR = "./banglabert_dialect_cls"

# =========================
# 2. Utils
# =========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def normalize_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)
    text = text.replace("৷", "।")
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

def normalize_label(label: str) -> str:
    if pd.isna(label):
        return ""
    return re.sub(r"\s+", " ", str(label)).strip()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

# =========================
# 3. Weighted trainer
# =========================
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask", None),
            token_type_ids=inputs.get("token_type_ids", None),
        )
        logits = outputs.get("logits")

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None
        )
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

# =========================
# 4. Load data
# =========================
set_seed(RANDOM_STATE)

df = pd.read_csv(CSV_PATH)
df[TEXT_COL] = df[TEXT_COL].apply(normalize_text)
df[LABEL_COL] = df[LABEL_COL].apply(normalize_label)

df = df[(df[TEXT_COL].str.len() > 0) & (df[LABEL_COL].str.len() > 0)].copy()
df = df.drop_duplicates(subset=[TEXT_COL, LABEL_COL]).reset_index(drop=True)

# remove ambiguous texts that appear with multiple labels
ambiguous = df.groupby(TEXT_COL)[LABEL_COL].nunique()
ambiguous = ambiguous[ambiguous > 1]
if len(ambiguous) > 0:
    df = df[~df[TEXT_COL].isin(set(ambiguous.index))].reset_index(drop=True)

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df[LABEL_COL],
    random_state=RANDOM_STATE,
)
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df[LABEL_COL],
    random_state=RANDOM_STATE,
)

le = LabelEncoder()
train_df["label"] = le.fit_transform(train_df[LABEL_COL])
valid_df["label"] = le.transform(valid_df[LABEL_COL])
test_df["label"] = le.transform(test_df[LABEL_COL])
num_labels = len(le.classes_)

# balanced class weights
class_counts = np.bincount(train_df["label"].values, minlength=num_labels)
class_weights = len(train_df) / (num_labels * np.maximum(class_counts, 1))
class_weights = torch.tensor(class_weights, dtype=torch.float)

train_ds = Dataset.from_pandas(
    train_df[[TEXT_COL, "label"]].rename(columns={TEXT_COL: "text"}),
    preserve_index=False
)
valid_ds = Dataset.from_pandas(
    valid_df[[TEXT_COL, "label"]].rename(columns={TEXT_COL: "text"}),
    preserve_index=False
)
test_ds = Dataset.from_pandas(
    test_df[[TEXT_COL, "label"]].rename(columns={TEXT_COL: "text"}),
    preserve_index=False
)

TypeError: Dataset.__init__() got an unexpected keyword argument 'preserve_index'

In [ ]:
# =========================
# 5. Tokenizer + tokenize
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

train_ds = train_ds.map(tokenize_fn, batched=True)
valid_ds = valid_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

columns_to_keep = ["input_ids", "attention_mask", "label"]
if "token_type_ids" in train_ds.column_names:
    columns_to_keep.append("token_type_ids")

train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in columns_to_keep])
valid_ds = valid_ds.remove_columns([c for c in valid_ds.column_names if c not in columns_to_keep])
test_ds = test_ds.remove_columns([c for c in test_ds.column_names if c not in columns_to_keep])

train_ds = train_ds.rename_column("label", "labels")
valid_ds = valid_ds.rename_column("label", "labels")
test_ds = test_ds.rename_column("label", "labels")

train_ds.set_format("torch")
valid_ds.set_format("torch")
test_ds.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# =========================
# 6. Model
# =========================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label={i: c for i, c in enumerate(le.classes_)},
    label2id={c: i for i, c in enumerate(le.classes_)},
)

# turn off fp16 if GPU is unavailable / incompatible
use_fp16 = torch.cuda.is_available()

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=use_fp16,
    report_to="none",
    save_total_limit=2,
    warmup_steps=100,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

In [ ]:
valid_metrics = trainer.evaluate(valid_ds)
print("Validation:", valid_metrics)

test_output = trainer.predict(test_ds)
test_preds = np.argmax(test_output.predictions, axis=1)

print("\n===== FINAL TEST RESULTS =====")
print("Test accuracy:", round(accuracy_score(test_df["label"], test_preds), 4))
print("Test macro-F1:", round(f1_score(test_df["label"], test_preds, average="macro"), 4))
print(classification_report(test_df["label"], test_preds, target_names=le.classes_))

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import joblib
joblib.dump(le, os.path.join(OUTPUT_DIR, "label_encoder.joblib"))
print(f"Saved model to: {OUTPUT_DIR}")

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/40549 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['electra.embeddings.LayerNorm.weight', 'electra.embeddings.LayerNorm.bias', 'electra.encoder.layer.0.attention.output.LayerNorm.weight', 'electra.encoder.layer.0.attention.output.LayerNorm.bias', 'electra.encoder.layer.0.output.LayerNorm.weight', 'electra.encoder.layer.0.output.LayerNorm.bias', 'electra.encoder.layer.1.attention.output.LayerNorm.weight', 'electra.encoder.layer.1.attention.output.LayerNorm.bias', 'electra.encoder.layer.1.output.LayerNorm.weight', 'electra.encoder.layer.1.output.LayerNorm.bias', 'electra.encoder.layer.2.attention.output.LayerNorm.weight', 'electra.encoder.layer.2.attention.output.LayerNorm.bias', 'electra.encoder.layer.2.output.LayerNorm.weight', 'electra.encoder.layer.2.output.LayerNorm.bias', 'electra.encoder.layer.3.attention.output.LayerNorm.weight', 'electra.encoder.layer.3.attention.output.LayerNorm.bias', 'electra.encoder.layer.3.output.LayerNorm.weight', 'electra.encoder.layer.3.output.Laye

Validation: {'eval_loss': 0.4632504880428314, 'eval_accuracy': 0.8907082264746499, 'eval_macro_f1': 0.8736436087736082, 'eval_runtime': 13.9824, 'eval_samples_per_second': 362.528, 'eval_steps_per_second': 11.371, 'epoch': 5.0}

===== FINAL TEST RESULTS =====
Test accuracy: 0.8921
Test macro-F1: 0.8741
                 precision    recall  f1-score   support

        Barisal       0.85      0.92      0.88       712
     Chittagong       0.97      0.94      0.95      1279
     Mymensingh       0.76      0.79      0.77       477
       Noakhali       0.83      0.84      0.83       794
Standard Bangla       0.94      0.92      0.93      1244
         Sylhet       0.90      0.85      0.88       563

       accuracy                           0.89      5069
      macro avg       0.87      0.88      0.87      5069
   weighted avg       0.89      0.89      0.89      5069



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to: ./banglabert_dialect_cls


In [6]:
import os
import re
import unicodedata
import joblib
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def normalize_text(text: str) -> str:
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)
    text = text.replace("৷", "।")
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

def load_dialect_model(model_dir: str):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    label_encoder = joblib.load(os.path.join(model_dir, "label_encoder.joblib"))

    model.to(device)
    model.eval()

    return {
        "tokenizer": tokenizer,
        "model": model,
        "label_encoder": label_encoder,
        "device": device,
    }

def predict_dialect(text: str, bundle, max_length: int = 128):
    clean_text = normalize_text(text)

    tokenizer = bundle["tokenizer"]
    model = bundle["model"]
    label_encoder = bundle["label_encoder"]
    device = bundle["device"]

    inputs = tokenizer(
        clean_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length,
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred_id = torch.argmax(probs, dim=1).item()
        confidence = probs[0, pred_id].item()

    dialect_name = label_encoder.inverse_transform([pred_id])[0]

    return {
        "text": text,
        "normalized_text": clean_text,
        "dialect": dialect_name,
        "label_id": pred_id,
        "confidence": confidence,
    }

model_dir = "/kaggle/working/banglabert_dialect_cls"   # change if needed

trained_model = load_dialect_model(model_dir)

result = predict_dialect("তুমারে দেখলাম", trained_model)
print(result["dialect"])
print(result["confidence"])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Sylhet
0.9960284233093262


In [5]:
import shutil

folder_to_zip = "/kaggle/working/banglabert_dialect_cls"   
zip_output_base = "/kaggle/working/banglabert_dialect_cls"

shutil.make_archive(zip_output_base, "zip", folder_to_zip)

print("Created:", zip_output_base + ".zip")

Created: /kaggle/working/banglabert_dialect_cls.zip
